# URL → Nine Ads (Krea 2 Turbo)

Paste a product URL → extract product data → generate **9 Meta/IG static ad concepts** with downloaded **Krea 2 Turbo** (open weights).

**No Magic Hour API. No Ideogram.**

### Before you run
1. Runtime → **Change runtime type** → **GPU** (T4 minimum; A100 better)
2. Accept access + license on [krea/Krea-2-Turbo](https://huggingface.co/krea/Krea-2-Turbo)
3. Add HF token: Colab **Secrets** → `HF_TOKEN` (or paste in the login cell)

### Demo URL (works without Cloudflare blocks)
`https://satoshi-demo.myshopify.com/products/classic-straight-jeans`

Optional harder test: Allbirds product pages (may block Colab IPs).


In [ ]:
# @title 1) Install
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Krea2Pipeline lives in recent diffusers (install from GitHub for safety)
pip_install(
    "git+https://github.com/huggingface/diffusers.git",
    "transformers>=4.51.0",
    "accelerate",
    "safetensors",
    "sentencepiece",
    "huggingface_hub",
    "Pillow",
    "beautifulsoup4",
    "lxml",
    "requests",
)
print("install ok")


In [ ]:
# @title 2) Hugging Face login (gated model)
import os
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    token = input("Paste HF token (read access): ").strip()

assert token, "HF_TOKEN required to download krea/Krea-2-Turbo"
login(token=token)
os.environ["HF_TOKEN"] = token
print("HF login ok")


In [ ]:
# @title 3) Load Krea 2 Turbo
import torch
from diffusers import Krea2Pipeline

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MODEL_ID = "krea/Krea-2-Turbo"

pipe = Krea2Pipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.enable_model_cpu_offload()  # T4-friendly

# Turbo distilled defaults
STEPS = 8
GUIDANCE = 0.0
SIZE = 1024
print(f"loaded {MODEL_ID} dtype={dtype} cuda={torch.cuda.get_device_name(0)}")


In [ ]:
# @title 4) Config — paste product URL
from pathlib import Path

PRODUCT_URL = "https://satoshi-demo.myshopify.com/products/classic-straight-jeans"
# PRODUCT_URL = "https://www.allbirds.com/products/mens-tree-gliders-natural-black-blizzard"  # may be blocked

PRIMARY_IMAGE_INDEX = 0
COMPOSITE_PRODUCT = True   # paste real product photo onto ad for identity lock
NUM_ADS = 9
SEED = 42
OUT_DIR = Path("url_ad_out")
OUT_DIR.mkdir(exist_ok=True)
print("URL:", PRODUCT_URL)


In [ ]:
# @title 5) Extract product (JSON-LD / OpenGraph — scraper_v10-lite)
import json, re, io
from typing import Any
import requests
from bs4 import BeautifulSoup
from PIL import Image
from IPython.display import display

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
FALLBACK_URL = "https://satoshi-demo.myshopify.com/products/classic-straight-jeans"


def _as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def _first_image(val) -> str | None:
    for item in _as_list(val):
        if isinstance(item, str) and item.startswith("http"):
            return item
        if isinstance(item, dict):
            u = item.get("url") or item.get("contentUrl")
            if isinstance(u, str) and u.startswith("http"):
                return u
    return None


def _price_from_offers(offers) -> str | None:
    for off in _as_list(offers):
        if not isinstance(off, dict):
            continue
        price = off.get("price") or off.get("lowPrice")
        cur = off.get("priceCurrency") or ""
        if price is not None:
            return f"{cur} {price}".strip()
    return None


def extract_product(url: str) -> dict[str, Any]:
    r = requests.get(url, headers=UA, timeout=25)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    brief = {
        "source_url": url,
        "product_name": None,
        "description": None,
        "price": None,
        "offer": None,
        "benefits": [],
        "features": [],
        "reviews": [],
        "images": [],
        "cta_candidates": ["Shop now", "Buy now"],
        "extraction_gaps": [],
    }

    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        try:
            data = json.loads(raw)
        except Exception:
            continue
        nodes = data if isinstance(data, list) else [data]
        for node in nodes:
            if not isinstance(node, dict):
                continue
            types = _as_list(node.get("@type"))
            types = [str(t).lower() for t in types]
            if "product" not in types:
                continue
            brief["product_name"] = brief["product_name"] or node.get("name")
            brief["description"] = brief["description"] or node.get("description")
            img = node.get("image")
            for im in _as_list(img):
                u = im if isinstance(im, str) else (im.get("url") if isinstance(im, dict) else None)
                if isinstance(u, str) and u.startswith("http") and u not in brief["images"]:
                    brief["images"].append(u)
            price = _price_from_offers(node.get("offers"))
            if price:
                brief["price"] = brief["price"] or price
            for rev in _as_list(node.get("review"))[:3]:
                if isinstance(rev, dict) and rev.get("reviewBody"):
                    brief["reviews"].append({
                        "quote": rev.get("reviewBody"),
                        "attribution": (rev.get("author") or {}).get("name") if isinstance(rev.get("author"), dict) else rev.get("author"),
                    })

    og_title = soup.find("meta", property="og:title")
    og_desc = soup.find("meta", property="og:description")
    og_img = soup.find("meta", property="og:image")
    if og_title and not brief["product_name"]:
        brief["product_name"] = og_title.get("content")
    if og_desc and not brief["description"]:
        brief["description"] = og_desc.get("content")
    if og_img:
        u = og_img.get("content")
        if u and u.startswith("http") and u not in brief["images"]:
            brief["images"].insert(0, u)

    # benefit-ish bullets
    for li in soup.select("li")[:40]:
        t = " ".join(li.get_text(" ", strip=True).split())
        if 20 <= len(t) <= 120 and t not in brief["benefits"]:
            brief["benefits"].append(t)
        if len(brief["benefits"]) >= 6:
            break

    if not brief["product_name"]:
        brief["extraction_gaps"].append("product_name")
    if not brief["images"]:
        brief["extraction_gaps"].append("images")
    if not brief["price"]:
        brief["extraction_gaps"].append("price")
    return brief


def download_image(url: str) -> Image.Image:
    resp = requests.get(url, headers=UA, timeout=30)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")


brief = None
last_err = None
for candidate in [PRODUCT_URL, FALLBACK_URL]:
    try:
        brief = extract_product(candidate)
        if brief.get("product_name") and brief.get("images"):
            PRODUCT_URL = candidate
            break
        last_err = f"thin extract for {candidate}: gaps={brief.get('extraction_gaps')}"
    except Exception as e:
        last_err = f"{candidate}: {e}"
        brief = None

assert brief and brief.get("images"), f"extract failed: {last_err}"

idx = max(0, min(PRIMARY_IMAGE_INDEX, len(brief["images"]) - 1))
product_img = download_image(brief["images"][idx])
product_img.save(OUT_DIR / "product_primary.jpg", quality=95)

print(json.dumps({k: brief[k] for k in ("product_name","price","description","images","benefits","reviews","extraction_gaps","source_url")}, indent=2)[:2500])
print("primary image:")
display(product_img.resize((384, 384)))


In [ ]:
# @title 6) Optional copy corrections (edit then re-run generate)
# Ground ad text ONLY in extracted / corrected fields — no invented claims.

NAME = brief.get("product_name") or "Product"
PRICE = brief.get("price") or ""
DESC = (brief.get("description") or "")[:280]
BENEFITS = brief.get("benefits") or []
REVIEWS = brief.get("reviews") or []
CTA = (brief.get("cta_candidates") or ["Shop now"])[0]

# Optional overrides:
# NAME = "Classic Straight Jeans"
# PRICE = "EUR 80"
# CTA = "Shop now"

benefit_line = BENEFITS[0] if BENEFITS else (DESC.split(".")[0] if DESC else NAME)
review_line = REVIEWS[0]["quote"] if REVIEWS else None
print("NAME:", NAME)
print("PRICE:", PRICE)
print("CTA:", CTA)
print("benefit:", benefit_line)
print("review:", (review_line or "")[:120])


In [ ]:
# @title 7) Build 9 angle prompts
ANGLES = [
    ("01_benefit", f"Feed ad for {NAME}. Big readable headline '{benefit_line[:48]}'. Small CTA button '{CTA}'. Show the exact product clearly. Clean modern Meta/Instagram square ad, premium lighting."),
    ("02_problem_solution", f"Square social ad. Headline 'Tired of settling?'. Subtext solving with {NAME}. CTA '{CTA}'. Product hero center. High-contrast commercial layout."),
    ("03_review", f"Testimonial ad for {NAME}. Quote text '{(review_line or 'Customers love the fit and feel.')[:90]}'. Product visible. Stars optional. CTA '{CTA}'."),
    ("04_feature", f"Feature callout ad for {NAME}. Three short callout labels from real product traits. Product large. CTA '{CTA}'. Minimal UI overlay."),
    ("05_comparison", f"Before/after style comparison ad focusing on choosing {NAME}. Clean split layout. CTA '{CTA}'. Product accurate."),
    ("06_offer", f"Offer ad for {NAME}. Headline includes exact price '{PRICE or 'Shop the drop'}'. Bold sale-style layout. CTA '{CTA}'. Product photo-real."),
    ("07_social_proof", f"Social proof ad for {NAME}. Headline 'Loved by customers'. Trust badges style. Product center. CTA '{CTA}'."),
    ("08_lifestyle", f"Lifestyle square ad: real-world scene featuring {NAME} naturally. Short headline '{NAME}'. CTA '{CTA}'. Cinematic but commercial."),
    ("09_minimal_hero", f"Minimal product hero ad on soft studio background. Tiny headline '{NAME}'. Tiny CTA '{CTA}'. Packaging and product identity preserved."),
]

assert len(ANGLES) == NUM_ADS
for k, p in ANGLES:
    print(k, "→", p[:100], "...")


In [ ]:
# @title 8) Generate 9 ads
import math
from PIL import ImageDraw, ImageFont

NEGATIVE = "blurry, low quality, watermark, illegible text, misspelled text, deformed product, extra logos"

def composite_product(ad: Image.Image, product: Image.Image) -> Image.Image:
    """Keep real product identity: place product photo on the generated ad."""
    out = ad.convert("RGBA")
    W, H = out.size
    pw = int(W * 0.42)
    prod = product.copy()
    prod.thumbnail((pw, pw), Image.Resampling.LANCZOS)
    px = (W - prod.width) // 2
    py = int(H * 0.38)
    shadow = Image.new("RGBA", out.size, (0, 0, 0, 0))
    sd = ImageDraw.Draw(shadow)
    sd.ellipse([px, py + prod.height - 12, px + prod.width, py + prod.height + 18], fill=(0, 0, 0, 60))
    out = Image.alpha_composite(out, shadow)
    out.paste(prod, (px, py))
    return out.convert("RGB")


ads = []
for i, (key, prompt) in enumerate(ANGLES):
    g = torch.Generator(device="cuda").manual_seed(SEED + i)
    kwargs = dict(
        prompt=prompt,
        height=SIZE,
        width=SIZE,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        generator=g,
    )
    if GUIDANCE > 0:
        kwargs["negative_prompt"] = NEGATIVE
    result = pipe(**kwargs)
    img = result.images[0]
    if COMPOSITE_PRODUCT:
        img = composite_product(img, product_img)
    path = OUT_DIR / f"{key}.png"
    img.save(path)
    ads.append((key, img))
    print("saved", path)

print("done", len(ads))



In [ ]:
# @title 9) Show 3x3 grid
from IPython.display import display

grid = Image.new("RGB", (SIZE * 3, SIZE * 3), (255, 255, 255))
for i, (key, img) in enumerate(ads):
    r, c = divmod(i, 3)
    grid.paste(img.resize((SIZE, SIZE)), (c * SIZE, r * SIZE))
grid_path = OUT_DIR / "grid_3x3.jpg"
grid.save(grid_path, quality=90)
print("grid:", grid_path)
display(grid.resize((768, 768)))
for key, img in ads:
    print(key)
    display(img.resize((320, 320)))


## Notes / roadblocks
- **Identity Edit LoRA** (`conradlocke/krea2-identity-edit`) needs ComfyUI dual-conditioning nodes — not in stock `Krea2Pipeline`. This notebook uses **Turbo T2I + optional product composite** so it runs without ComfyUI errors.
- Accept HF gate for `krea/Krea-2-Turbo` or download fails.
- Prefer **A100/L4**; T4 may be slow / tight VRAM (cpu offload helps).
- Brand PDPs (Allbirds/Amazon) often block datacenter IPs — demo Shopify URL is the reliable paste target.
- No Ideogram (local open weights lack product-edit conditioning).
